# 02 — Classes and Memory

## Why This Matters for DSA
Linked data structures require us to define custom node objects containing both data and links. In C++, we do this using **Structs** and **Classes**. Additionally, since these structures grow and shrink dynamically, we cannot rely on static stack-allocated variables. We must allocate nodes in **Heap** memory dynamically and free them when they are removed. Understanding structs, constructors, destructors, and memory management is the key to writing leak-free, clean DSA code.

## Table of Contents
1. Structs vs. Classes in C++
2. Constructors and Member Initializer Lists
3. Dynamic Memory Allocation (`new` and `delete`)
4. Memory Layout — Stack vs. Heap
5. Checkpoint & Dynamic Array Class Example


## 1. Structs vs. Classes in C++

In C++, both `struct` and `class` can hold data members and member functions. The only difference is the default access specifier:
- **Struct:** Members are **public** by default. In DSA, we use structs for simple data-carrying nodes (e.g. `TreeNode`, `ListNode`).
- **Class:** Members are **private** by default. We use classes for full data structures where we want to encapsulate implementation details and expose a public API (e.g. `Trie`, `AvlTree`).

In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

// Struct: public by default
struct Node {
    int value; // public
};

// Class: private by default
class MyData {
    int secretValue; // private
public:
    void setSecret(int val) { secretValue = val; }
    int getSecret() { return secretValue; }
};

int main() {
    Node node;
    node.value = 100; // Accessible directly
    
    MyData data;
    // data.secretValue = 200; // Error! private member
    data.setSecret(200);
    cout << "node.value: " << node.value << "\n";
    cout << "secretVal:  " << data.getSecret() << "\n";
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Constructors and Member Initializer Lists

- **Constructors:** Special member functions called automatically when an object is instantiated. Used to initialize variables.
- **Member Initializer Lists:** The most efficient way to initialize members in C++. Written after the constructor parameter list with a colon, e.g. `Node(int x) : value(x), left(nullptr) {}`.

In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

struct TreeNode {
    int value;
    TreeNode* left;
    TreeNode* right;
    
    // Constructor utilizing Member Initializer List
    TreeNode(int val) : value(val), left(nullptr), right(nullptr) {
        cout << "TreeNode containing " << value << " constructed!\n";
    }
};

int main() {
    TreeNode node(10); // constructor called
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Dynamic Memory Allocation (`new` and `delete`)

Standard variables are declared locally and destroyed automatically when functions exit. To create persistent objects that survive beyond function boundaries, we allocate memory dynamically:
- **`new` Operator:** Allocates memory on the heap and returns its pointer.
- **`delete` Operator:** Frees the heap memory. Failing to delete allocated memory causes **memory leaks**, consuming RAM until the program runs out of space.
- **Destructors:** A class destructor (`~ClassName()`) is automatically invoked when the object scope ends, and is where we write clean-up logic to release dynamically allocated memory.

In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

int main() {
    // Allocate an integer on the Heap
    int* heapPtr = new int(42);
    
    cout << "Address on Heap: " << heapPtr << "\n";
    cout << "Value on Heap:   " << *heapPtr << "\n";
    
    // Free memory
    delete heapPtr;
    heapPtr = nullptr; // Avoid dangling pointer
    
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Memory Layout — Stack vs. Heap

C++ divides RAM into regions, each with different performance characteristics:
1. **The Stack:**
   - Stored in contiguous memory.
   - Allocation and deallocation are managed automatically by the compiler.
   - Fast but limited in size. Local variables reside here.
2. **The Heap:**
   - Large, unstructured pool of memory.
   - Allocation and deallocation are managed manually by the developer (`new`/`delete`).
   - Slightly slower access but holds persistent data. Linked list and tree nodes reside here.

| Feature | The Stack | The Heap |
|---|---|---|
| **Allocation** | Automatic | Manual (`new`) |
| **Deallocation** | Automatic (when scope ends) | Manual (`delete`) |
| **Size** | Small (typically 1-8 MB) | Large (virtual memory limit) |
| **Speed** | Very Fast | Slower (requires pointer hops) |
| **Structure** | LIFO (Call frames) | No structural order |

## 5. Checkpoint & Dynamic Array Class Example

### Checkpoint Questions
1. Why does class node deletion require a recursive postorder cleanup function?
2. What happens if you allocate memory using `new` but forget to call `delete`?
3. What is a member initializer list and why is it preferred over assignment inside the constructor body?

### Answer Key
1. Because a node has child pointers. If you delete a parent node first, its child pointers are lost in memory, making it impossible to access or delete the child subtrees (causing memory leaks). A postorder cleanup deletes the children *first* and then deletes the parent.
2. A memory leak occurs. The memory remains occupied by the program even if no pointers reference it, which can eventually crash the application when memory runs out.
3. A member initializer list initializes variables directly when memory is allocated for the object, avoiding the default construction and subsequent assignment steps, which is faster.

### Dynamic Array Class Example
Let's write a simple class that allocates an array on the Heap, writes to it, and frees the memory in its destructor.

In [ ]:
%%writefile temp.cpp
#include <iostream>
using namespace std;

class DynamicArray {
private:
    int* arr;
    int size;
public:
    // Constructor allocates heap memory
    DynamicArray(int s) : size(s) {
        arr = new int[size];
        for (int i = 0; i < size; i++) arr[i] = 0;
        cout << "Array of size " << size << " allocated on the heap.\n";
    }

    // Destructor frees heap memory
    ~DynamicArray() {
        delete[] arr;
        cout << "Array memory released!\n";
    }

    void set(int idx, int val) {
        if (idx >= 0 && idx < size) arr[idx] = val;
    }

    int get(int idx) const {
        if (idx >= 0 && idx < size) return arr[idx];
        return -1;
    }
};

int main() {
    {
        DynamicArray myArr(5);
        myArr.set(0, 10);
        myArr.set(1, 20);
        cout << "Value at index 0: " << myArr.get(0) << "\n";
        cout << "Value at index 1: " << myArr.get(1) << "\n";
    } // myArr goes out of scope here; destructor is called automatically
    
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp